In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from torch.distributions import Uniform, Normal

---

In [ ]:
X, y = make_moons(n_samples=2000, noise=0.08, random_state=42)

In [ ]:
X -= np.mean(X, axis=0)
X /= np.std(X)

In [ ]:
plt.figure(figsize=(8, 6))
plt.grid(alpha=0.5, zorder=1)

plt.scatter(X[y==0, 0], X[y==0, 1], zorder=2)
plt.scatter(X[y==1, 0], X[y==1, 1], zorder=2)

plt.show()

In [ ]:
def beta(t, beta_min=0.1, beta_max=20):
    return beta_min + t * (beta_max - beta_min)

def m(t, beta_min=0.1, beta_max=20):
    return np.exp(-0.5 * (beta_min * t + 0.5 * (beta_max - beta_min) * t**2))

def sigma(t, beta_min=0.1, beta_max=20):
    return np.sqrt(1 - m(t, beta_min=beta_min, beta_max=beta_max)**2)

In [ ]:
T = np.linspace(1e-3, 1, num=100)

fig, axs = plt.subplots(1, 3, figsize=(16, 5))

axs[0].grid(alpha=0.5, zorder=1)
axs[0].plot(T, beta(T), label=r'$\beta(t)$', zorder=2)
axs[0].legend()

axs[1].grid(alpha=0.5, zorder=1)
axs[1].plot(T, m(T), label=r'$m(t)$', zorder=2)
axs[1].legend()

axs[2].grid(alpha=0.5, zorder=1)
axs[2].plot(T, sigma(T), label=r'$\sigma(t)$', zorder=2)
axs[2].legend()

plt.show()

In [ ]:
fig, axs = plt.subplots(1, 6, figsize=(16, 4))

for i, t in enumerate([0, 0.1, 0.3, 0.5, 0.7, 1]):
    eps = np.random.normal(loc=0.0, scale=1.0, size=X.shape)
    xt = m(t) * X + sigma(t) * eps

    axs[i].grid(alpha=0.5, zorder=1)
    axs[i].scatter(xt[y==0, 0], xt[y==0, 1], s=5, alpha=0.5, zorder=2)
    axs[i].scatter(xt[y==1, 0], xt[y==1, 1], s=5, alpha=0.5, zorder=2)

    axs[i].set_title(f't={t}')
    axs[i].set_xlim(-3.5, 3.5)
    axs[i].set_ylim(-3.5, 3.5)
    axs[i].set_aspect('equal', 'box')

In [ ]:
xt.mean()

In [ ]:
xt.var()

---

## Solving SDE numericaly

$$dx = -\tfrac12 \beta(t)\, x\, dt + \sqrt{\beta(t)}\, dW_t,\qquad t\in[0,1]$$

Let us look at one small SDE step. This is an ordinary Euler-Maruyama step:

$$x_{\text{new}} = x - \tfrac12\beta\,x\,dt + \sqrt{\beta\,dt}\;z,\qquad z\sim\mathcal N(0,1).$$

Which can be written analytically as

$$x_t = m_t x_0 + \sigma_t\varepsilon,$$
where
$$\beta(t) = \beta_{\min} + t(\beta_{\max}-\beta_{\min}),\qquad \beta_{\min}=0.1, \, \beta_{\max}=20$$
$$m_t = \exp\!\Big(-\tfrac12\!\int_0^t \beta(s)\,ds\Big),\qquad \sigma_t^2 = 1 - m_t^2 .$$

In [ ]:
M = 500
x0 = np.zeros((M, 2))
x0[:, 0] += 1
xt_history = [x0]
N = 500
dt = 1 / 500

for i in range(N):
    t = dt * (i + 1)
    x = xt_history[-1]
    z = np.random.normal(loc=0.0, scale=1.0, size=x.shape)
    xt_history.append(x - 1/2 * beta(t) * x * dt + np.sqrt(beta(t) * dt) * z)

xt_history = np.array(xt_history)

In [ ]:
xt_history.shape

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))
axs[0].grid(alpha=0.5, zorder=1)

t = np.linspace(0, 1, N+1)

for i in range(M):
    axs[0].plot(t, xt_history[:, i, 0], alpha=0.1, color='C0')

axs[0].plot(t, m(t) * x0[0, 0], color='C1')
axs[0].plot(t, m(t) * x0[0, 0] + sigma(t), color='C2')
axs[0].plot(t, m(t) * x0[0, 0] - sigma(t), color='C2')
axs[0].set_ylim(-4.5, 4.5)

axs[1].hist(xt_history[-1, :, 0], orientation='horizontal', bins=30, color='C0', edgecolor='black', alpha=0.7)
axs[1].set_ylim(-4.5, 4.5)

plt.show()

---

In [ ]:
N_small = 30
idx = np.random.choice(len(X), size=N_small, replace=False)
data = X[idx]

fig, ax = plt.subplots(figsize=(8, 6))
ax.grid(alpha=0.5, zorder=1)
ax.scatter(X[:, 0], X[:, 1], s=5, alpha=0.2, zorder=2)
ax.scatter(data[:, 0], data[:, 1], s=30, color='red', zorder=3)

plt.show()

In [ ]:
def dist2(x, t, data):
    # squared distance from each point x to the center of each cloud m(t) * x_i
    # x: (B, 2), data: (N, 2), result: (B, N)
    d2 = np.zeros((len(x), len(data)))
    for i in range(len(data)):
        d2[:, i] = np.sum((x - m(t) * data[i])**2, axis=1)
    return d2


def exact_eps(x, t, data):
    a = -dist2(x, t, data) / (2 * sigma(t)**2)

    # parent weights. Subtract the maximum, otherwise at small t exp(a) gives 0 everywhere
    a = a - a.max(axis=1, keepdims=True)
    w = np.exp(a)
    w = w / w.sum(axis=1, keepdims=True)    # (B, N)

    D = w @ data                            # (B, 2) best guess of the parent
    eps = (x - m(t) * D) / sigma(t)         # the noise that turns m*D into x
    return eps, D, w


def exact_score(x, t, data):
    eps, D, w = exact_eps(x, t, data)
    return -eps / sigma(t)

In [ ]:
def log_p(x, t, data):
    a = -dist2(x, t, data) / (2 * sigma(t)**2)
    amax = a.max(axis=1, keepdims=True)
    log_sum = amax[:, 0] + np.log(np.sum(np.exp(a - amax), axis=1))
    return log_sum - np.log(len(data)) - np.log(2 * np.pi * sigma(t)**2)

In [ ]:
x = np.random.normal(size=(1000, 2)) * 1.5
h = 1e-5
dx = np.array([h, 0])
dy = np.array([0, h])

for t in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]:
    s = exact_score(x, t, data)
    gx = (log_p(x + dx, t, data) - log_p(x - dx, t, data)) / (2 * h)
    gy = (log_p(x + dy, t, data) - log_p(x - dy, t, data)) / (2 * h)
    g = np.stack([gx, gy], axis=1)
    err = np.abs(g - s).max() / np.abs(s).max()
    print(f't={t}: relative error {err:.1e}, mean |s| = {np.linalg.norm(s, axis=1).mean():.1f}')

In [ ]:
x_probe = np.array([[0.0, 0.0]])

fig, axs = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for i, t in enumerate([0.05, 0.3, 1.0]):
    eps, D, w = exact_eps(x_probe, t, data)
    axs[i].grid(alpha=0.5, zorder=1)
    axs[i].bar(range(N_small), w[0], zorder=2)
    axs[i].set_title(f't={t}')
    axs[i].set_xlabel('dataset point index')

axs[0].set_ylabel('parent weight')
plt.show()

In [ ]:
g = np.linspace(-3, 3, 25)
GX, GY = np.meshgrid(g, g)
grid = np.stack([GX.ravel(), GY.ravel()], axis=1)

fig, axs = plt.subplots(1, 3, figsize=(18, 6))

for i, t in enumerate([0.05, 0.3, 1.0]):
    s = exact_score(grid, t, data)
    s = s / np.linalg.norm(s, axis=1, keepdims=True)   # direction only

    axs[i].grid(alpha=0.5, zorder=1)
    axs[i].quiver(grid[:, 0], grid[:, 1], s[:, 0], s[:, 1], zorder=2)
    axs[i].scatter(data[:, 0], data[:, 1], s=15, color='red', alpha=0.3, zorder=3)
    axs[i].scatter(m(t) * data[:, 0], m(t) * data[:, 1], s=25, color='orange', edgecolor='k', zorder=4)

    axs[i].set_title(f't={t}')
    axs[i].set_xlim(-3.5, 3.5)
    axs[i].set_ylim(-3.5, 3.5)
    axs[i].set_aspect('equal', 'box')

plt.show()

In [ ]:
s1 = exact_score(grid, 1.0, data)
print('max|s + x| =', np.abs(s1 + grid).max())
print('m(1) =', m(1.0))

---

## Train an MLIP to predict $\nabla p_t(x)$

In [ ]:
def beta(t, beta_min=0.1, beta_max=20):
    return (beta_min + t * (beta_max - beta_min))

def m(t, beta_min=0.1, beta_max=20):
    return torch.exp(-0.5 * (beta_min * t + 0.5 * (beta_max - beta_min) * t**2))

def sigma(t, beta_min=0.1, beta_max=20):
    return torch.sqrt(1 - m(t, beta_min=beta_min, beta_max=beta_max)**2)

In [ ]:
X, Y = make_moons(n_samples=2000, noise=0.08, random_state=42)
X -= np.mean(X, axis=0)
X /= np.std(X)
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X, Y, train_size=0.8, random_state=42)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

In [ ]:
freqs = torch.exp(torch.linspace(0, np.log(30), 16, device=device))

def time_emb(x, t):
    # t: (B, 1) -> (B, 32)

    return torch.cat((torch.sin(t * freqs), torch.cos(t * freqs)), dim=1)

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()

        self.m = nn.Sequential(
            nn.Linear(2 + 32, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x, t):
        emb = time_emb(x, t)
        x = torch.concat((x, emb), dim=1)
        return self.m(x)


In [ ]:
model = Model().to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 10000
# ITERS = 10
BATCH = 512
VERBOSE = 100

t_min, t_max = 1e-3, 1
uniform_dist = Uniform(low=1e-3, high=1)
normal_dist = Normal(loc=0, scale=1)

loss_train_history = []
loss_test_history = []

for epoch in range(EPOCHS):
    train_loss_acc = 0
    test_loss_acc = 0

    # Train
    sampler = torch.randperm(len(X_train))
    for i in range(len(X_train) // BATCH):
        optimizer.zero_grad()
        idx = sampler[i*BATCH:(i+1)*BATCH]

        x = X_train[idx]

        t = t_min + (1 - t_min) * torch.rand(BATCH, 1, device=device)
        eps = normal_dist.sample(x.shape).to(device)

        xt = m(t) * x + sigma(t) * eps

        pred = model(xt, t)
        train_loss = ((pred - eps)**2).mean()
        train_loss.backward()

        optimizer.step()

        train_loss_acc += train_loss.item()

    with torch.no_grad():
        # Test
        t = t_min + (1 - t_min) * torch.rand(len(X_test), 1, device=device)
        eps = normal_dist.sample(X_test.shape).to(device)
        xt = m(t) * X_test + sigma(t) * eps

        pred = model(xt, t)
        test_loss = ((pred - eps)**2).mean()

    if epoch % VERBOSE == 0:
        loss_train_history.append(train_loss_acc / (len(X_train) // BATCH))
        loss_test_history.append(test_loss.item())

    print(f"Epoch: {epoch}/{EPOCHS}")
    print(f'Train loss: {loss_train_history[-1]}')
    print(f'Test  loss: {loss_test_history[-1]}')


In [ ]:
plt.figure(figsize=(8, 6))
plt.grid(alpha=0.5, zorder=1)

plt.plot(loss_train_history, label='train')
plt.plot(loss_test_history, label='test')

plt.legend()

In [ ]:
model.eval()

ts = np.linspace(t_min, 1, 30)
losses = []
x0 = X_test.repeat(10, 1)      # 4000 points, to make the curve smooth

with torch.no_grad():
    for t_val in ts:
        t = torch.full((len(x0), 1), float(t_val), device=device)
        eps = torch.randn_like(x0)
        xt = m(t) * x0 + sigma(t) * eps
        pred = model(xt, t)
        losses.append(((pred - eps)**2).mean().item())

print('mean over t:', np.mean(losses))

plt.figure(figsize=(8, 6))
plt.grid(alpha=0.5, zorder=1)
plt.plot(ts, losses, 'o-', zorder=2)
plt.xlabel('t')
plt.ylabel('loss')
plt.show()

In [ ]:
g = torch.linspace(-3, 3, 25)
GX, GY = torch.meshgrid(g, g, indexing='xy')
grid = torch.stack([GX.ravel(), GY.ravel()], dim=1).to(device)

x_data = X_train.cpu().numpy()
gr = grid.cpu().numpy()

fig, axs = plt.subplots(1, 3, figsize=(18, 6))

with torch.no_grad():
    for i, t_val in enumerate([0.05, 0.3, 1.0]):
        t = torch.full((len(grid), 1), t_val, device=device)
        s = -model(grid, t) / sigma(t)                # score
        s = s / s.norm(dim=1, keepdim=True)           # direction only
        s = s.cpu().numpy()
        mt = m(t)[0, 0].item()

        axs[i].grid(alpha=0.5, zorder=1)
        axs[i].scatter(mt * x_data[:, 0], mt * x_data[:, 1], s=3, color='C1', alpha=0.3, zorder=2)
        axs[i].quiver(gr[:, 0], gr[:, 1], s[:, 0], s[:, 1], zorder=3)

        axs[i].set_title(f't={t_val}')
        axs[i].set_xlim(-3.5, 3.5)
        axs[i].set_ylim(-3.5, 3.5)
        axs[i].set_aspect('equal', 'box')

plt.show()

---

In [ ]:
def sample(model, x_start, K=500, mode='ode'):
    # x_start: (n, 2) starting noise at t = 1
    # mode: 'ode' or 'sde'
    model.eval()
    dt = (1 - t_min) / K
    x = x_start.clone()
    history = [x.cpu().numpy()]

    with torch.no_grad():
        for k in range(K):
            t = torch.full((len(x), 1), 1 - k * dt, device=device)
            s = -model(x, t) / sigma(t)                   # score

            if mode == 'ode':
                x = x + 0.5 * beta(t) * (x + s) * dt
            else:
                x = x + 0.5 * beta(t) * (x + 2 * s) * dt
                if k < K - 1:                             # no noise at the last step
                    x = x + torch.sqrt(beta(t) * dt) * torch.randn_like(x)

            history.append(x.cpu().numpy())

    return x.cpu().numpy(), np.array(history)

In [ ]:
n = 2000
K = 500
x_start = torch.randn(n, 2, device=device)

x_ode, hist_ode = sample(model, x_start, K=K, mode='ode')
x_sde, hist_sde = sample(model, x_start, K=K, mode='sde')
x_data = X_train.cpu().numpy()

fig, axs = plt.subplots(1, 3, figsize=(18, 6))

for i, (pts, title) in enumerate(zip([x_data, x_ode, x_sde], ['data', 'ODE', 'SDE'])):
    axs[i].grid(alpha=0.5, zorder=1)
    axs[i].scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5, zorder=2)
    axs[i].set_title(title)
    axs[i].set_xlim(-3, 3)
    axs[i].set_ylim(-3, 3)
    axs[i].set_aspect('equal', 'box')

plt.show()

In [ ]:
t_grid = 1 - np.arange(K + 1) * (1 - t_min) / K

fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for j, (hist, title) in enumerate(zip([hist_ode, hist_sde], ['ODE', 'SDE'])):
    axs[j].grid(alpha=0.5, zorder=1)
    for i in range(20):
        axs[j].plot(t_grid, hist[:, i, 0], alpha=0.7, zorder=2)
    axs[j].invert_xaxis()              # generation time runs left to right, from 1 to 0
    axs[j].set_xlabel('t')
    axs[j].set_title(title)

axs[0].set_ylabel('x coordinate')
plt.show()

In [ ]:
a, _ = sample(model, x_start, K=200, mode='ode')
b, _ = sample(model, x_start, K=200, mode='ode')
print('ODE, two runs from the same start:', np.abs(a - b).max())

a, _ = sample(model, x_start, K=200, mode='sde')
b, _ = sample(model, x_start, K=200, mode='sde')
print('SDE, two runs from the same start:', np.abs(a - b).max())

In [ ]:
Ks = [5, 10, 20, 50, 500]

fig, axs = plt.subplots(2, len(Ks), figsize=(4 * len(Ks), 8))

for j, K_ in enumerate(Ks):
    for i, mode in enumerate(['ode', 'sde']):
        pts, _ = sample(model, x_start, K=K_, mode=mode)
        axs[i, j].grid(alpha=0.5, zorder=1)
        axs[i, j].scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.5, zorder=2)
        axs[i, j].set_title(f'{mode.upper()}, K={K_}')
        axs[i, j].set_xlim(-3, 3)
        axs[i, j].set_ylim(-3, 3)
        axs[i, j].set_aspect('equal', 'box')

plt.show()

---

In [ ]:
def drift(x, t_val):
    # ODE right-hand side when moving backward: x <- x + h * drift
    t = torch.full((len(x), 1), float(t_val), device=device)
    s = -model(x, t) / sigma(t)
    return 0.5 * beta(t) * (x + s)


def time_grid(K, rho=1.0):
    # K steps from t = 1 to t = t_min, for rho > 1 the steps are denser at small t
    u = np.linspace(1, 0, K + 1)
    return t_min + (1 - t_min) * u**rho

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.grid(alpha=0.5, zorder=1)

for i, rho in enumerate([1, 2, 3]):
    ts = time_grid(20, rho)
    ax.scatter(ts, np.full_like(ts, i), s=20, zorder=2)

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['rho=1', 'rho=2', 'rho=3'])
ax.set_xlabel('t')
plt.show()

In [ ]:
def sample_ode(x_start, ts, method='euler'):
    model.eval()
    x = x_start.clone()
    history = [x.cpu().numpy()]
    nfe = 0

    with torch.no_grad():
        for k in range(len(ts) - 1):
            t, t_next = ts[k], ts[k + 1]
            h = t - t_next

            d = drift(x, t)
            nfe += 1

            if method == 'euler':
                x = x + h * d
            else:
                x_euler = x + h * d                # trial step
                d_next = drift(x_euler, t_next)    # slope at the end of the step
                nfe += 1
                x = x + h * 0.5 * (d + d_next)     # average slope

            history.append(x.cpu().numpy())

    return x.cpu().numpy(), np.array(history), nfe

In [ ]:
x_start = torch.randn(2000, 2, device=device)

# reference: Heun with a very small step
x_ref, _, _ = sample_ode(x_start, time_grid(1000), 'heun')

nfes = [10, 20, 40, 80, 160, 320]

fig, ax = plt.subplots(figsize=(8, 6))
ax.grid(alpha=0.5, zorder=1, which='both')

for rho, ls in zip([1, 2, 3], ['-', '--', ':']):
    for method, color in zip(['euler', 'heun'], ['C0', 'C1']):
        errs = []
        for nfe in nfes:
            K = nfe if method == 'euler' else nfe // 2
            x, _, _ = sample_ode(x_start, time_grid(K, rho), method)
            errs.append(np.sqrt(((x - x_ref)**2).sum(axis=1)).mean())
        ax.loglog(nfes, errs, ls, color=color, marker='o', label=f'{method}, rho={rho}', zorder=2)

ax.set_xlabel('number of network calls (NFE)')
ax.set_ylabel('mean error relative to the reference')
ax.legend()
plt.show()

In [ ]:
rho_good = 3
nfes_show = [10, 20, 40]
rows = [('euler', 1), ('heun', 1), ('euler', rho_good), ('heun', rho_good)]

fig, axs = plt.subplots(len(rows), len(nfes_show), figsize=(4 * len(nfes_show), 4 * len(rows)))

for i, (method, rho) in enumerate(rows):
    for j, nfe in enumerate(nfes_show):
        K = nfe if method == 'euler' else nfe // 2
        x, _, _ = sample_ode(x_start, time_grid(K, rho), method)

        axs[i, j].grid(alpha=0.5, zorder=1)
        axs[i, j].scatter(x[:, 0], x[:, 1], s=2, alpha=0.5, zorder=2)
        axs[i, j].set_title(f'{method}, rho={rho}, NFE={nfe}')
        axs[i, j].set_xlim(-3, 3)
        axs[i, j].set_ylim(-3, 3)
        axs[i, j].set_aspect('equal', 'box')

plt.show()

In [ ]:
model.eval()
t_fine = torch.linspace(t_min, 1, 5000, device=device)[:, None]
pts = torch.tensor([[0.0, 0.0], [1.0, 0.5], [-1.5, 0.5]], device=device)

fig, axs = plt.subplots(2, len(pts), figsize=(18, 9))

with torch.no_grad():
    for i, p in enumerate(pts):
        x = p.repeat(len(t_fine), 1)
        eps = model(x, t_fine).cpu().numpy()
        tt = t_fine.cpu().numpy()[:, 0]

        for row in range(2):
            axs[row, i].grid(alpha=0.5, zorder=1)
            axs[row, i].plot(tt, eps[:, 0], label=r'$\varepsilon_x$', zorder=2)
            axs[row, i].plot(tt, eps[:, 1], label=r'$\varepsilon_y$', zorder=2)
            axs[row, i].legend()

        axs[0, i].set_title(f'x = {p.tolist()}')
        axs[1, i].set_xlim(0.30, 0.36)       # magnifier
        axs[1, i].set_title('zoomed in')

plt.show()